In [18]:
from pathlib import Path
import copy
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.ensemble import RandomForestRegressor
from joblib import Parallel, delayed


def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


SEED = 42
set_seed(SEED, deterministic=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch version:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
TRAFFIC_CSV = Path("cleaned_traffic_data.csv")
META_XLSX   = Path("pems_output.xlsx")

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS_DIR = OUT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Fixed experimental setup
# ---------------------------------------------------------
TRAIN_END = pd.Timestamp("2024-11-15 23:59:59")
VAL_END   = pd.Timestamp("2024-11-30 23:59:59")

IN_LEN = 24
OUT_LEN = 72
EVAL_HORIZONS = [1, 6, 12, 24]

COVERAGE_THRESHOLD = 0.98
K_NEIGHBORS = 2

TAG = "eval_h1_6_12_24_keep_out72"

RESULTS_SUMMARY = OUT_DIR / f"results_summary_{TAG}.csv"
MAE_TABLE_PATH = OUT_DIR / f"mae_table_{TAG}.csv"
RMSE_TABLE_PATH = OUT_DIR / f"rmse_table_{TAG}.csv"

print("TRAFFIC_CSV:", TRAFFIC_CSV)
print("META_XLSX:", META_XLSX)
print("Training target length OUT_LEN:", OUT_LEN)
print("Reported horizons:", EVAL_HORIZONS)
print("RESULTS_SUMMARY:", RESULTS_SUMMARY)

Torch version: 2.1.1+cu121
Device: cuda
GPU: Quadro P5000
TRAFFIC_CSV: cleaned_traffic_data.csv
META_XLSX: pems_output.xlsx
Training target length OUT_LEN: 72
Reported horizons: [1, 6, 12, 24]
RESULTS_SUMMARY: artifacts/results_summary_eval_h1_6_12_24_keep_out72.csv


In [14]:
PROJECT_ROOT = Path(".").resolve()

strict_matches = sorted(PROJECT_ROOT.rglob("pems_graph_dataset_strict.npz"))
base_matches   = sorted(PROJECT_ROOT.rglob("pems_graph_dataset.npz"))

DATASET_STRICT = strict_matches[0] if len(strict_matches) > 0 else (OUT_DIR / "pems_graph_dataset_strict.npz")
DATASET_BASE   = base_matches[0]   if len(base_matches)   > 0 else (OUT_DIR / "pems_graph_dataset.npz")

print("Project root:", PROJECT_ROOT)
print("Found strict artifacts:")
for p in strict_matches:
    print(" -", p)
print("Found base artifacts:")
for p in base_matches:
    print(" -", p)


def require_col(df: pd.DataFrame, candidates, friendly_name: str):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not find column for '{friendly_name}'. Tried: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


def to_datetime_safe(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")


def pct_missing(s: pd.Series) -> float:
    return float(s.isna().mean() * 100.0)


def make_time_features(timestamps: pd.DatetimeIndex) -> np.ndarray:
    hours = timestamps.hour.values
    dow   = timestamps.dayofweek.values
    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    dow_sin  = np.sin(2 * np.pi * dow / 7.0)
    dow_cos  = np.cos(2 * np.pi * dow / 7.0)
    return np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1).astype(np.float32)


def build_adjacency_from_metadata(meta_df: pd.DataFrame, stations: np.ndarray, k_neighbors: int = 2) -> np.ndarray:
    id_col = "station"

    abs_pm_col = None
    for cand in ["Abs PM", "abs_pm", "AbsPM", "Postmile", "PM"]:
        if cand in meta_df.columns:
            abs_pm_col = cand
            break

    fwy_col2 = None
    for cand in ["Fwy", "FWY", "fwy", "Freeway"]:
        if cand in meta_df.columns:
            fwy_col2 = cand
            break

    if abs_pm_col is None or fwy_col2 is None:
        raise KeyError(
            f"Metadata missing Abs PM or Fwy columns. Found AbsPM={abs_pm_col}, Fwy={fwy_col2}"
        )

    meta_sub = meta_df[meta_df[id_col].isin(stations)].copy()
    meta_sub["abs_pm"] = pd.to_numeric(meta_sub[abs_pm_col], errors="coerce")
    meta_sub["fwy"] = meta_sub[fwy_col2].astype(str)

    station_to_idx = {s: i for i, s in enumerate(stations)}
    N_local = len(stations)
    A = np.zeros((N_local, N_local), dtype=np.float32)

    all_dists = []
    for fwy, grp in meta_sub.sort_values(["fwy", "abs_pm"]).groupby("fwy"):
        pm = grp["abs_pm"].dropna().values
        if len(pm) < 2:
            continue
        d = np.diff(np.sort(pm))
        d = d[d > 0]
        all_dists.extend(d.tolist())

    sigma = float(np.median(all_dists)) if len(all_dists) else 0.5
    sigma = max(sigma, 1e-3)

    def w(dist):
        return float(np.exp(- (dist ** 2) / (sigma ** 2)))

    for fwy, grp in meta_sub.sort_values(["fwy", "abs_pm"]).groupby("fwy"):
        grp = grp.dropna(subset=["abs_pm"]).sort_values("abs_pm")
        ids = grp[id_col].astype(int).tolist()
        pms = grp["abs_pm"].astype(float).tolist()

        for i, sid in enumerate(ids):
            ii = station_to_idx[sid]
            for step in range(1, k_neighbors + 1):
                if i - step >= 0:
                    sj = ids[i - step]
                    jj = station_to_idx[sj]
                    A[ii, jj] = w(abs(pms[i] - pms[i - step]))
                if i + step < len(ids):
                    sj = ids[i + step]
                    jj = station_to_idx[sj]
                    A[ii, jj] = w(abs(pms[i] - pms[i + step]))

    np.fill_diagonal(A, 1.0)
    A = np.maximum(A, A.T)
    return A


def make_strict_dataset(base_npz: Path, strict_npz: Path, train_end: pd.Timestamp, val_end: pd.Timestamp):
    d = np.load(base_npz, allow_pickle=True)

    X = d["X"]
    Y = d["Y"]
    A = d["A"]
    stations = d["stations"]
    timestamps = pd.to_datetime(d["timestamps"])

    in_len = int(np.array(d["in_len"]).item())
    out_len = int(np.array(d["out_len"]).item())

    flow_mean = d["flow_mean"]
    flow_std  = d["flow_std"]
    speed_mean = d["speed_mean"]
    speed_std  = d["speed_std"]

    T_total = X.shape[0]
    max_t = T_total - (in_len + out_len) + 1
    starts = np.arange(max_t, dtype=np.int32)

    out_start_times = timestamps[starts + in_len]
    out_end_times   = timestamps[starts + in_len + out_len - 1]

    train_starts = starts[out_end_times <= train_end]
    val_starts   = starts[(out_start_times > train_end) & (out_end_times <= val_end)]
    test_starts  = starts[out_start_times > val_end]

    np.savez_compressed(
        strict_npz,
        X=X.astype(np.float32),
        Y=Y.astype(np.float32),
        A=A.astype(np.float32),
        stations=stations.astype(np.int32),
        timestamps=np.array(timestamps.astype("datetime64[ns]")),
        train_starts=train_starts,
        val_starts=val_starts,
        test_starts=test_starts,
        in_len=np.array([in_len], dtype=np.int32),
        out_len=np.array([out_len], dtype=np.int32),
        flow_mean=flow_mean.astype(np.float32),
        flow_std=flow_std.astype(np.float32),
        speed_mean=speed_mean.astype(np.float32),
        speed_std=speed_std.astype(np.float32),
        seed=np.array([SEED], dtype=np.int32),
    )
    print("Saved strict dataset:", strict_npz)


def build_base_dataset_from_raw(
    traffic_csv: Path,
    meta_xlsx: Path,
    base_npz: Path,
    train_end: pd.Timestamp,
    val_end: pd.Timestamp,
    in_len: int,
    out_len: int,
    coverage_threshold: float = 0.98,
    k_neighbors: int = 2,
):
    assert traffic_csv.exists(), f"Missing {traffic_csv}"
    assert meta_xlsx.exists(), f"Missing {meta_xlsx}"

    traffic_raw = pd.read_csv(traffic_csv)
    meta_raw = pd.read_excel(meta_xlsx)

    # --- Identify traffic columns robustly ---
    ts_col   = require_col(traffic_raw, ["Timestamp", "timestamp", "Time", "Datetime"], "Timestamp")
    st_col   = require_col(traffic_raw, ["Station", "station", "ID"], "Station ID")
    flow_col = require_col(traffic_raw, ["Total Flow", "total_flow", "Flow", "total flow"], "Total Flow")
    spd_col  = require_col(traffic_raw, ["Avg Speed", "avg_speed", "Speed", "Avg speed"], "Avg Speed")

    lane_col = require_col(traffic_raw, ["Lane Type", "lane_type", "LaneType"], "Lane Type")
    dir_col  = require_col(traffic_raw, ["Direction of Travel", "direction", "Dir"], "Direction")
    dist_col = require_col(traffic_raw, ["District", "district"], "District")

    traffic = traffic_raw.rename(columns={
        ts_col: "timestamp",
        st_col: "station",
        flow_col: "total_flow",
        spd_col: "avg_speed",
        lane_col: "lane_type",
        dir_col: "direction",
        dist_col: "district",
    }).copy()

    traffic["timestamp"] = to_datetime_safe(traffic["timestamp"])
    traffic["station"] = pd.to_numeric(traffic["station"], errors="coerce").astype("Int64")
    traffic = traffic.dropna(subset=["timestamp", "station"]).copy()
    traffic["station"] = traffic["station"].astype(int)

    # --- Standardize metadata ---
    meta_id_col = require_col(meta_raw, ["ID", "station", "Station"], "Meta Station ID")
    meta = meta_raw.rename(columns={meta_id_col: "station"}).copy()
    meta["station"] = pd.to_numeric(meta["station"], errors="coerce").astype("Int64")
    meta = meta.dropna(subset=["station"]).copy()
    meta["station"] = meta["station"].astype(int)

    # --- Merge ---
    df = traffic.merge(meta, on="station", how="inner", validate="m:1")

    # --- Deduplicate if necessary ---
    dup_count = df.duplicated(subset=["timestamp", "station"]).sum()
    print("Duplicate (timestamp, station) rows:", int(dup_count))

    if dup_count > 0:
        df = (
            df.groupby(["timestamp", "station"], as_index=False)
              .agg({
                  "total_flow": "sum",
                  "avg_speed": "mean",
                  "lane_type": "first",
                  "direction": "first",
                  "district": "first",
                  **{c: "first" for c in meta.columns if c != "station"}
              })
        )

    # --- Full timestamp index ---
    all_times = pd.DatetimeIndex(sorted(df["timestamp"].unique()))
    T_total = len(all_times)

    # --- Coverage-based station selection ---
    counts = df.groupby("station")["timestamp"].nunique()
    coverage = counts / T_total
    keep_stations = coverage[coverage >= coverage_threshold].index

    df2 = df[df["station"].isin(keep_stations)].copy()
    stations = np.array(sorted(df2["station"].unique()), dtype=int)
    N_local = len(stations)

    print(f"Timestamps (T) = {T_total}")
    print(f"Stations kept (N) = {N_local} (coverage threshold={coverage_threshold})")

    # --- Build matrices ---
    flow = (
        df2.pivot(index="timestamp", columns="station", values="total_flow")
           .reindex(index=all_times, columns=stations)
           .sort_index()
    )

    speed = (
        df2.pivot(index="timestamp", columns="station", values="avg_speed")
           .reindex(index=all_times, columns=stations)
           .sort_index()
    )

    print("Flow matrix:", flow.shape)
    print("Speed matrix:", speed.shape)
    print("Flow missing fraction:", float(np.isnan(flow.to_numpy()).mean()))
    print("Speed missing fraction:", float(np.isnan(speed.to_numpy()).mean()))

    # --- Leakage-safe imputation ---
    meta_type_col = None
    for cand in ["Type", "type", "Station Type"]:
        if cand in df2.columns:
            meta_type_col = cand
            break

    fwy_col = None
    for cand in ["Fwy", "FWY", "fwy", "Freeway"]:
        if cand in df2.columns:
            fwy_col = cand
            break

    if meta_type_col is None or fwy_col is None:
        raise KeyError(
            f"Missing metadata columns for speed lookup. Found meta_type={meta_type_col}, fwy={fwy_col}"
        )

    train_time_mask = flow.index <= train_end

    flow_ff = flow.ffill()
    flow_train_mean_station = flow_ff.loc[train_time_mask].mean(axis=0)
    flow_train_mean_global = flow_ff.loc[train_time_mask].stack().mean()
    flow_imp = flow_ff.fillna(flow_train_mean_station).fillna(flow_train_mean_global)

    train_rows = df2[df2["timestamp"] <= train_end].copy()
    train_rows["hour"] = train_rows["timestamp"].dt.hour

    speed_grp_cols = ["lane_type", meta_type_col, "hour", fwy_col, "district"]
    speed_lookup = train_rows.groupby(speed_grp_cols)["avg_speed"].mean()
    global_speed_train_mean = train_rows["avg_speed"].mean()

    station_info = (
        df2.groupby("station")
           .agg(
               lane_type=("lane_type", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
               meta_type=(meta_type_col, lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
               fwy=(fwy_col, lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
               district=("district", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
           )
           .reindex(stations)
    )

    speed_ff = speed.ffill()
    speed_np = speed_ff.to_numpy(dtype=np.float32)
    miss = np.isnan(speed_np)
    hours = speed_ff.index.hour.values

    for j, st in enumerate(stations):
        if not miss[:, j].any():
            continue

        info = station_info.loc[st]
        lane_type = info["lane_type"]
        meta_type = info["meta_type"]
        fwy = info["fwy"]
        district = info["district"]

        idxs = np.where(miss[:, j])[0]
        fill_vals = []
        for t_idx in idxs:
            h = int(hours[t_idx])
            key = (lane_type, meta_type, h, fwy, district)
            fill_vals.append(speed_lookup.get(key, np.nan))

        speed_np[idxs, j] = np.array(fill_vals, dtype=np.float32)

    speed_imp = pd.DataFrame(speed_np, index=speed_ff.index, columns=speed_ff.columns)
    speed_train_mean_station = speed_imp.loc[train_time_mask].mean(axis=0)
    speed_imp = speed_imp.fillna(speed_train_mean_station).fillna(global_speed_train_mean)

    print("After imputation:")
    print("Flow missing fraction:", float(np.isnan(flow_imp.to_numpy()).mean()))
    print("Speed missing fraction:", float(np.isnan(speed_imp.to_numpy()).mean()))

    # --- Build tensors ---
    time_feats = make_time_features(flow_imp.index)
    time_feats_b = np.repeat(time_feats[:, None, :], repeats=N_local, axis=1)

    flow_arr = flow_imp.to_numpy(dtype=np.float32)[:, :, None]
    speed_arr = speed_imp.to_numpy(dtype=np.float32)[:, :, None]

    X = np.concatenate([flow_arr, speed_arr, time_feats_b], axis=2).astype(np.float32)
    Y = flow_arr.squeeze(-1).astype(np.float32)

    print("X shape:", X.shape)
    print("Y shape:", Y.shape)

    # --- Static adjacency for saved base artifact ---
    meta_for_adj = meta.copy()
    meta_for_adj["station"] = meta_for_adj["station"].astype(int)
    A = build_adjacency_from_metadata(meta_for_adj, stations=stations, k_neighbors=k_neighbors)

    print("A shape:", A.shape)
    print("Adjacency density:", float((A > 0).mean()))

    # --- Original base split by output start ---
    timestamps = pd.DatetimeIndex(flow_imp.index)
    max_t = X.shape[0] - (in_len + out_len) + 1
    starts = np.arange(max_t, dtype=np.int32)

    out_start_times = timestamps[starts + in_len]
    train_starts = starts[out_start_times <= train_end]
    val_starts   = starts[(out_start_times > train_end) & (out_start_times <= val_end)]
    test_starts  = starts[out_start_times > val_end]

    flow_mean = X[train_time_mask, :, 0].mean(axis=0).astype(np.float32)
    flow_std  = (X[train_time_mask, :, 0].std(axis=0) + 1e-6).astype(np.float32)
    speed_mean = X[train_time_mask, :, 1].mean(axis=0).astype(np.float32)
    speed_std  = (X[train_time_mask, :, 1].std(axis=0) + 1e-6).astype(np.float32)

    np.savez_compressed(
        base_npz,
        X=X.astype(np.float32),
        Y=Y.astype(np.float32),
        A=A.astype(np.float32),
        stations=stations.astype(np.int32),
        timestamps=np.array(timestamps.astype("datetime64[ns]")),
        train_starts=train_starts,
        val_starts=val_starts,
        test_starts=test_starts,
        in_len=np.array([in_len], dtype=np.int32),
        out_len=np.array([out_len], dtype=np.int32),
        flow_mean=flow_mean.astype(np.float32),
        flow_std=flow_std.astype(np.float32),
        speed_mean=speed_mean.astype(np.float32),
        speed_std=speed_std.astype(np.float32),
        seed=np.array([SEED], dtype=np.int32),
    )

    print("Saved base dataset:", base_npz)


if DATASET_STRICT.exists():
    print("\nUsing existing strict artifact:")
    print(DATASET_STRICT)

elif DATASET_BASE.exists():
    print("\nStrict artifact not found, but base artifact exists.")
    print("Base artifact:", DATASET_BASE)
    print("Recreating strict artifact...")
    make_strict_dataset(DATASET_BASE, OUT_DIR / "pems_graph_dataset_strict.npz", TRAIN_END, VAL_END)
    DATASET_STRICT = OUT_DIR / "pems_graph_dataset_strict.npz"

else:
    print("\nNeither strict nor base artifact was found.")
    print("Rebuilding base artifact from raw files, then creating strict artifact...")

    build_base_dataset_from_raw(
        traffic_csv=TRAFFIC_CSV,
        meta_xlsx=META_XLSX,
        base_npz=OUT_DIR / "pems_graph_dataset.npz",
        train_end=TRAIN_END,
        val_end=VAL_END,
        in_len=IN_LEN,
        out_len=OUT_LEN,
        coverage_threshold=COVERAGE_THRESHOLD,
        k_neighbors=K_NEIGHBORS,
    )

    make_strict_dataset(
        base_npz=OUT_DIR / "pems_graph_dataset.npz",
        strict_npz=OUT_DIR / "pems_graph_dataset_strict.npz",
        train_end=TRAIN_END,
        val_end=VAL_END,
    )

    DATASET_BASE = OUT_DIR / "pems_graph_dataset.npz"
    DATASET_STRICT = OUT_DIR / "pems_graph_dataset_strict.npz"

print("\nFinal strict dataset path:")
print(DATASET_STRICT)
print("Exists:", DATASET_STRICT.exists())

Project root: /notebooks
Found strict artifacts:
 - /notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset_strict.npz
Found base artifacts:
 - /notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset.npz

Using existing strict artifact:
/notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset_strict.npz

Final strict dataset path:
/notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/artifacts/pems_graph_dataset_strict.npz
Exists: True


## Load the strict dataset and create shared scaled tensors

From this point onward, all models use the same strict artifact.

The strict artifact contains:

- the feature tensor `X`
- the flow target tensor `Y`
- the saved adjacency matrix `A`
- the strict train, validation, and test windows
- train-only normalization statistics

The model training task remains unchanged:

- input length `24`
- output length `72`

Only the reported horizons differ from what we had initially.

In [15]:
assert DATASET_STRICT.exists(), f"Missing strict dataset: {DATASET_STRICT}"

data = np.load(DATASET_STRICT, allow_pickle=True)

X = data["X"].astype(np.float32)          # (T, N, F)
Y = data["Y"].astype(np.float32)          # (T, N)
A = data["A"].astype(np.float32)          # (N, N)

stations = data["stations"]
timestamps = pd.to_datetime(data["timestamps"])

train_starts = data["train_starts"].astype(np.int64)
val_starts   = data["val_starts"].astype(np.int64)
test_starts  = data["test_starts"].astype(np.int64)

artifact_in_len  = int(np.array(data["in_len"]).item())
artifact_out_len = int(np.array(data["out_len"]).item())

flow_mean  = data["flow_mean"].astype(np.float32)
flow_std   = data["flow_std"].astype(np.float32)
speed_mean = data["speed_mean"].astype(np.float32)
speed_std  = data["speed_std"].astype(np.float32)

flow_std  = np.maximum(flow_std, 1e-6).astype(np.float32)
speed_std = np.maximum(speed_std, 1e-6).astype(np.float32)

assert artifact_in_len == IN_LEN, f"Expected IN_LEN={IN_LEN}, found {artifact_in_len}"
assert artifact_out_len == OUT_LEN, f"Expected OUT_LEN={OUT_LEN}, found {artifact_out_len}"

T, N, Fdim = X.shape

print("Loaded strict dataset successfully.")
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("A shape:", A.shape)
print("Stations:", N)
print("Feature dimension:", Fdim)
print("Train windows:", len(train_starts))
print("Validation windows:", len(val_starts))
print("Test windows:", len(test_starts))


def time_encoding(dt_index: pd.DatetimeIndex) -> np.ndarray:
    hours = dt_index.hour.values
    dow   = dt_index.dayofweek.values

    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    dow_sin  = np.sin(2 * np.pi * dow / 7.0)
    dow_cos  = np.cos(2 * np.pi * dow / 7.0)

    return np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1).astype(np.float32)


TF_all = time_encoding(pd.DatetimeIndex(timestamps))  # (T, 4)

X_scaled = X.copy()
X_scaled[:, :, 0] = (X_scaled[:, :, 0] - flow_mean[None, :]) / flow_std[None, :]
X_scaled[:, :, 1] = (X_scaled[:, :, 1] - speed_mean[None, :]) / speed_std[None, :]

Y_scaled = (Y - flow_mean[None, :]) / flow_std[None, :]

X_fnt = np.transpose(X_scaled, (2, 1, 0)).copy()  # (F, N, T)

print("TF_all shape:", TF_all.shape)
print("X_fnt shape:", X_fnt.shape)
print("Scaled target mean:", float(Y_scaled.mean()))
print("Scaled target std:", float(Y_scaled.std()))

Loaded strict dataset successfully.
X shape: (2208, 1821, 6)
Y shape: (2208, 1821)
A shape: (1821, 1821)
Stations: 1821
Feature dimension: 6
Train windows: 1009
Validation windows: 289
Test windows: 673
TF_all shape: (2208, 4)
X_fnt shape: (6, 1821, 2208)
Scaled target mean: -781.403564453125
Scaled target std: 30704.76953125


## Shared dataset, dataloaders, evaluation, and saving utilities

All deep models are trained from the same sliding-window dataset.

Each sample contains:

- `x`: history tensor of shape `(F, N, IN_LEN)`
- `y`: future target tensor of shape `(OUT_LEN, N)`
- `tf`: future time features of shape `(OUT_LEN, 4)`

The evaluation procedure computes MAE and RMSE in original traffic-flow units at the selected report horizons:

- `1h`
- `6h`
- `12h`
- `24h`

In [16]:
class ForecastWindowDataset(Dataset):
    def __init__(self, X_fnt, Y_scaled, TF_all, starts, in_len, out_len):
        self.X_fnt = X_fnt
        self.Y = Y_scaled
        self.TF = TF_all
        self.starts = np.asarray(starts, dtype=np.int64)
        self.in_len = int(in_len)
        self.out_len = int(out_len)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = int(self.starts[idx])

        x = self.X_fnt[:, :, s:s + self.in_len]
        y = self.Y[s + self.in_len:s + self.in_len + self.out_len, :]
        tf = self.TF[s + self.in_len:s + self.in_len + self.out_len, :]

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(y).float(),
            torch.from_numpy(tf).float(),
        )


train_ds = ForecastWindowDataset(X_fnt, Y_scaled, TF_all, train_starts, IN_LEN, OUT_LEN)
val_ds   = ForecastWindowDataset(X_fnt, Y_scaled, TF_all, val_starts, IN_LEN, OUT_LEN)
test_ds  = ForecastWindowDataset(X_fnt, Y_scaled, TF_all, test_starts, IN_LEN, OUT_LEN)

BATCH_SIZE = 8

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

xb, yb, tfb = next(iter(train_loader))
print("Batch x shape:", xb.shape)
print("Batch y shape:", yb.shape)
print("Batch tf shape:", tfb.shape)


flow_mean_t = torch.tensor(flow_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
flow_std_t  = torch.tensor(flow_std, dtype=torch.float32, device=DEVICE).view(1, 1, -1)


def print_metrics(title: str, metrics: dict):
    print("\n" + title)
    for h in sorted(metrics.keys()):
        print(f"  {h:>3}h | MAE={metrics[h]['MAE']:.3f} | RMSE={metrics[h]['RMSE']:.3f}")


def avg_mae(metrics: dict) -> float:
    return float(np.mean([metrics[h]["MAE"] for h in metrics]))


@torch.inference_mode()
def eval_horizons_fast(model, loader, eval_horizons=EVAL_HORIZONS):
    model.eval()
    acc = {h: {"abs": 0.0, "sq": 0.0, "count": 0} for h in eval_horizons}

    for xb, yb, tfb in tqdm(loader, desc="Eval", leave=False):
        xb  = xb.to(DEVICE, non_blocking=True)
        yb  = yb.to(DEVICE, non_blocking=True)
        tfb = tfb.to(DEVICE, non_blocking=True)

        pred = model(xb, tfb)

        pred_u = pred * flow_std_t + flow_mean_t
        true_u = yb   * flow_std_t + flow_mean_t

        for h in eval_horizons:
            idx = h - 1
            err = pred_u[:, idx, :] - true_u[:, idx, :]
            acc[h]["abs"] += float(err.abs().sum())
            acc[h]["sq"]  += float((err ** 2).sum())
            acc[h]["count"] += err.numel()

    metrics = {}
    for h in eval_horizons:
        mae = acc[h]["abs"] / acc[h]["count"]
        rmse = (acc[h]["sq"] / acc[h]["count"]) ** 0.5
        metrics[h] = {"MAE": float(mae), "RMSE": float(rmse)}
    return metrics


def make_run_dir(model_name: str, tag: str = TAG) -> Path:
    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    run_dir = RUNS_DIR / f"{stamp}_{model_name}_{tag}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def save_metrics_files(run_dir: Path, split_name: str, metrics: dict):
    save_json(run_dir / f"{split_name}_metrics.json", metrics)
    rows = [{"horizon_h": h, "MAE": metrics[h]["MAE"], "RMSE": metrics[h]["RMSE"]} for h in sorted(metrics)]
    pd.DataFrame(rows).to_csv(run_dir / f"{split_name}_metrics.csv", index=False)


def append_results_summary(model_name: str, run_dir: Path, test_metrics: dict):
    row = {
        "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model_name": model_name,
        "run_dir": str(run_dir),
        "tag": TAG,
    }
    for h in EVAL_HORIZONS:
        row[f"test_MAE_{h}h"] = float(test_metrics[h]["MAE"])
        row[f"test_RMSE_{h}h"] = float(test_metrics[h]["RMSE"])

    df_new = pd.DataFrame([row])

    if RESULTS_SUMMARY.exists():
        df_old = pd.read_csv(RESULTS_SUMMARY)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new

    df.to_csv(RESULTS_SUMMARY, index=False)
    return RESULTS_SUMMARY


@torch.inference_mode()
def collect_selected_horizon_predictions(model, loader, horizons=EVAL_HORIZONS):
    model.eval()
    pred_list = []
    true_list = []

    for xb, yb, tfb in tqdm(loader, desc="Collect preds", leave=False):
        xb  = xb.to(DEVICE, non_blocking=True)
        yb  = yb.to(DEVICE, non_blocking=True)
        tfb = tfb.to(DEVICE, non_blocking=True)

        pred = model(xb, tfb)

        pred_u = pred * flow_std_t + flow_mean_t
        true_u = yb   * flow_std_t + flow_mean_t

        pred_sel = []
        true_sel = []
        for h in horizons:
            idx = h - 1
            pred_sel.append(pred_u[:, idx, :].detach().cpu().numpy())
            true_sel.append(true_u[:, idx, :].detach().cpu().numpy())

        pred_sel = np.stack(pred_sel, axis=1)  # (B, H, N)
        true_sel = np.stack(true_sel, axis=1)

        pred_list.append(pred_sel.astype(np.float32))
        true_list.append(true_sel.astype(np.float32))

    pred_all = np.concatenate(pred_list, axis=0)
    true_all = np.concatenate(true_list, axis=0)
    return pred_all, true_all


def train_deep_model(
    model_name: str,
    model: nn.Module,
    train_loader,
    val_loader,
    test_loader,
    epochs: int = 40,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    clip: float = 5.0,
    patience: int = 6,
    eval_every: int = 2,
    save_predictions: bool = True,
):
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    config = {
        "model_name": model_name,
        "seed": SEED,
        "in_len": IN_LEN,
        "out_len": OUT_LEN,
        "eval_horizons": EVAL_HORIZONS,
        "epochs": epochs,
        "lr": lr,
        "weight_decay": weight_decay,
        "clip": clip,
        "patience": patience,
        "eval_every": eval_every,
        "tag": TAG,
    }
    save_json(run_dir / "config.json", config)

    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_score = float("inf")
    best_state = None
    best_epoch = None
    bad_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0

        for xb, yb, tfb in tqdm(train_loader, desc=f"Train {epoch}/{epochs}", leave=False):
            xb  = xb.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE, non_blocking=True)
            tfb = tfb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb, tfb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()

            running += float(loss.item())

        row = {"epoch": epoch, "train_loss": running / max(len(train_loader), 1)}

        if (epoch % eval_every == 0) or (epoch == epochs):
            val_metrics = eval_horizons_fast(model, val_loader)
            val_avg = avg_mae(val_metrics)

            row["val_avg_MAE"] = float(val_avg)
            for h in EVAL_HORIZONS:
                row[f"val_MAE_{h}h"] = float(val_metrics[h]["MAE"])
                row[f"val_RMSE_{h}h"] = float(val_metrics[h]["RMSE"])

            print(f"\nEpoch {epoch}: train_loss={row['train_loss']:.6f} | val_avg_MAE={val_avg:.3f}")
            print_metrics("Validation", val_metrics)

            if val_avg < best_score:
                best_score = float(val_avg)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                torch.save(best_state, run_dir / "best.pt")
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"\nEarly stopping triggered. Best val_avg_MAE={best_score:.3f} at epoch {best_epoch}.")
                    history.append(row)
                    break

        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

    assert best_state is not None, "Best model was never saved."

    model.load_state_dict(best_state)

    print("\nEvaluating on test set...")
    test_metrics = eval_horizons_fast(model, test_loader)
    print_metrics(f"{model_name} — TEST", test_metrics)

    save_metrics_files(run_dir, "test", test_metrics)
    append_results_summary(model_name, run_dir, test_metrics)

    save_json(run_dir / "training_summary.json", {
        "best_val_avg_MAE": best_score,
        "best_epoch": best_epoch,
        "test_metrics": test_metrics,
    })

    if save_predictions:
        pred_u, true_u = collect_selected_horizon_predictions(model, test_loader, horizons=EVAL_HORIZONS)
        np.savez_compressed(
            run_dir / "test_pred_true_selected_horizons.npz",
            pred=pred_u.astype(np.float32),
            true=true_u.astype(np.float32),
            horizons=np.array(EVAL_HORIZONS, dtype=np.int32),
            stations=stations,
        )

    print("Saved outputs to:", run_dir)
    return run_dir, test_metrics

Batch x shape: torch.Size([8, 6, 1821, 24])
Batch y shape: torch.Size([8, 72, 1821])
Batch tf shape: torch.Size([8, 72, 4])


## Classical baselines: Elastic Net and Random Forest

The classical baselines follow the same high-level strategy as the original project:

- fit one model per node
- use the recent history of that node as handcrafted predictors
- append future calendar features for the selected report horizons
- predict those report horizons directly as a multi-output regression problem

These baselines provide a strong non-neural reference point in the final comparison.

In [7]:
H_OFF = np.array([h - 1 for h in EVAL_HORIZONS], dtype=np.int64)
H = len(EVAL_HORIZONS)


def node_features_and_targets(node: int, starts: np.ndarray):
    Xn = X_scaled[:, node, :]  # (T, F)

    win = np.lib.stride_tricks.sliding_window_view(Xn, window_shape=IN_LEN, axis=0)
    X_hist = win[starts].reshape(len(starts), -1)

    idx = starts[:, None] + IN_LEN + H_OFF[None, :]
    X_tf = TF_all[idx].reshape(len(starts), -1)

    X_feat = np.concatenate([X_hist, X_tf], axis=1)
    y = Y_scaled[idx, node]

    return X_feat.astype(np.float32), y.astype(np.float32)


def save_classical_run_outputs(model_name: str, run_dir: Path, pred_u: np.ndarray, true_u: np.ndarray, metrics: dict):
    save_metrics_files(run_dir, "test", metrics)
    append_results_summary(model_name, run_dir, metrics)

    np.savez_compressed(
        run_dir / "test_pred_true_selected_horizons.npz",
        pred=pred_u.astype(np.float32),
        true=true_u.astype(np.float32),
        horizons=np.array(EVAL_HORIZONS, dtype=np.int32),
        stations=stations,
    )


def run_elasticnet_baseline(alpha=1e-3, l1_ratio=0.5, jobs=4):
    model_name = "ElasticNet"
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    save_json(run_dir / "config.json", {
        "model_name": model_name,
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "jobs": jobs,
        "eval_horizons": EVAL_HORIZONS,
        "tag": TAG,
    })

    S_test = len(test_starts)
    pred_scaled = np.zeros((S_test, N, H), dtype=np.float32)
    true_scaled = np.zeros((S_test, N, H), dtype=np.float32)

    def fit_one(node: int):
        Xtr, ytr = node_features_and_targets(node, train_starts)
        Xte, yte = node_features_and_targets(node, test_starts)

        mdl = make_pipeline(
            StandardScaler(),
            MultiTaskElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=5000,
                random_state=SEED,
            ),
        )
        mdl.fit(Xtr, ytr)
        pred = mdl.predict(Xte).astype(np.float32)
        return node, pred, yte

    results = Parallel(n_jobs=jobs, prefer="threads")(
        delayed(fit_one)(node) for node in tqdm(range(N), desc="ElasticNet per-node")
    )

    for node, pred, yte in results:
        pred_scaled[:, node, :] = pred
        true_scaled[:, node, :] = yte

    pred_u = pred_scaled * flow_std[None, :, None] + flow_mean[None, :, None]
    true_u = true_scaled * flow_std[None, :, None] + flow_mean[None, :, None]

    metrics = {}
    for j, h in enumerate(EVAL_HORIZONS):
        err = pred_u[:, :, j] - true_u[:, :, j]
        metrics[h] = {
            "MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
        }

    print_metrics("Elastic Net — TEST", metrics)
    save_classical_run_outputs(model_name, run_dir, pred_u, true_u, metrics)
    return run_dir, metrics


def run_random_forest_baseline(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=5,
    max_features="sqrt",
    jobs=4,
):
    model_name = "RandomForest"
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    save_json(run_dir / "config.json", {
        "model_name": model_name,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
        "jobs": jobs,
        "eval_horizons": EVAL_HORIZONS,
        "tag": TAG,
    })

    S_test = len(test_starts)
    pred_scaled = np.zeros((S_test, N, H), dtype=np.float32)
    true_scaled = np.zeros((S_test, N, H), dtype=np.float32)

    def fit_one(node: int):
        Xtr, ytr = node_features_and_targets(node, train_starts)
        Xte, yte = node_features_and_targets(node, test_starts)

        mdl = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            n_jobs=1,
            random_state=SEED,
        )
        mdl.fit(Xtr, ytr)
        pred = mdl.predict(Xte).astype(np.float32)
        return node, pred, yte

    results = Parallel(n_jobs=jobs, prefer="threads")(
        delayed(fit_one)(node) for node in tqdm(range(N), desc="RF per-node")
    )

    for node, pred, yte in results:
        pred_scaled[:, node, :] = pred
        true_scaled[:, node, :] = yte

    pred_u = pred_scaled * flow_std[None, :, None] + flow_mean[None, :, None]
    true_u = true_scaled * flow_std[None, :, None] + flow_mean[None, :, None]

    metrics = {}
    for j, h in enumerate(EVAL_HORIZONS):
        err = pred_u[:, :, j] - true_u[:, :, j]
        metrics[h] = {
            "MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
        }

    print_metrics("Random Forest — TEST", metrics)
    save_classical_run_outputs(model_name, run_dir, pred_u, true_u, metrics)
    return run_dir, metrics

In [8]:
run_dir_en, test_en = run_elasticnet_baseline(alpha=1e-3, l1_ratio=0.5, jobs=4)
print("Elastic Net run dir:", run_dir_en)

run_dir_rf, test_rf = run_random_forest_baseline(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=5,
    max_features="sqrt",
    jobs=4,
)
print("Random Forest run dir:", run_dir_rf)

Run dir: artifacts/runs/20260320_233642_ElasticNet_eval_h1_6_12_24_keep_out72


ElasticNet per-node:   0%|          | 0/1821 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.2229136228561401, tolerance: 0.4010547399520874
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.4256468713283539, tolerance: 0.40035566687583923
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.9521648287773132, tolerance: 0.4009798467159271
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:241


Elastic Net — TEST
    1h | MAE=87.306 | RMSE=164.194
    6h | MAE=148.227 | RMSE=301.406
   12h | MAE=151.116 | RMSE=306.920
   24h | MAE=148.615 | RMSE=303.706
Elastic Net run dir: artifacts/runs/20260320_233642_ElasticNet_eval_h1_6_12_24_keep_out72
Run dir: artifacts/runs/20260320_234454_RandomForest_eval_h1_6_12_24_keep_out72


RF per-node:   0%|          | 0/1821 [00:00<?, ?it/s]


Random Forest — TEST
    1h | MAE=101.119 | RMSE=228.188
    6h | MAE=108.060 | RMSE=247.700
   12h | MAE=113.183 | RMSE=260.996
   24h | MAE=114.145 | RMSE=258.278
Random Forest run dir: artifacts/runs/20260320_234454_RandomForest_eval_h1_6_12_24_keep_out72


## Graph supports and GraphWaveNet family

The graph models use a refined direction-aware spatial structure built from:

- station identifiers
- direction of travel
- freeway grouping
- absolute postmile proximity

This support construction is then used by the GraphWaveNet family:

- GraphWaveNet
- GraphWaveNet-GRU
- GraphWaveNet-LSTM
- GraphWaveNet-GRU-LSTM

The encoder remains graph-temporal, while the recurrent variants refine the node-wise temporal embeddings before the multi-horizon output head.

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path(".").resolve()

def find_first_match(patterns):
    matches = []
    for pat in patterns:
        matches.extend(PROJECT_ROOT.rglob(pat))
    matches = sorted(set(matches))
    return matches

traffic_matches = find_first_match([
    "cleaned_traffic_data.csv",
    "*cleaned*traffic*.csv",
    "*traffic*.csv",
])

meta_matches = find_first_match([
    "pems_output.xlsx",
    "*pems*.xlsx",
    "*output*.xlsx",
])

TRAFFIC_CSV = traffic_matches[0] if len(traffic_matches) > 0 else None
META_XLSX = meta_matches[0] if len(meta_matches) > 0 else None

print("Resolved TRAFFIC_CSV:", TRAFFIC_CSV)
print("Resolved META_XLSX:", META_XLSX)


def require_col(df: pd.DataFrame, candidates, friendly_name: str):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not find column for '{friendly_name}'. Tried: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


def row_normalize_dense(A_dense, eps=1e-6):
    d = A_dense.sum(axis=1, keepdims=True)
    return A_dense / (d + eps)


def dense_to_sparse(A_dense, device):
    idx = np.nonzero(A_dense)
    values = A_dense[idx].astype(np.float32)
    indices = np.vstack(idx)

    return torch.sparse_coo_tensor(
        torch.tensor(indices, dtype=torch.long, device=device),
        torch.tensor(values, dtype=torch.float32, device=device),
        size=A_dense.shape,
        device=device,
    ).coalesce()


USE_REFINED_SUPPORTS = (
    TRAFFIC_CSV is not None and TRAFFIC_CSV.exists() and
    META_XLSX is not None and META_XLSX.exists()
)

if USE_REFINED_SUPPORTS:
    print("\nUsing refined raw-file graph support construction...")

    traffic_raw = pd.read_csv(TRAFFIC_CSV)
    meta_raw = pd.read_excel(META_XLSX)

    traffic_raw.columns = [str(c).strip() for c in traffic_raw.columns]
    meta_raw.columns = [str(c).strip() for c in meta_raw.columns]

    st_col_traffic = require_col(
        traffic_raw,
        ["Station", "station", "ID"],
        "traffic station id"
    )
    dir_col_traffic = require_col(
        traffic_raw,
        ["Direction of Travel", "direction", "Dir"],
        "traffic direction"
    )

    tmp = traffic_raw[[st_col_traffic, dir_col_traffic]].copy()
    tmp = tmp.rename(columns={st_col_traffic: "station", dir_col_traffic: "direction"})
    tmp["station"] = pd.to_numeric(tmp["station"], errors="coerce").astype("Int64")
    tmp = tmp.dropna(subset=["station"])
    tmp["station"] = tmp["station"].astype(int)

    station_dir = tmp.groupby("station")["direction"].agg(
        lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]
    )
    station_dir = station_dir.reindex(stations)

    meta_id_col = require_col(meta_raw, ["station", "ID", "Station"], "metadata station id")
    meta_fwy_col = require_col(meta_raw, ["Fwy", "FWY", "fwy", "Freeway"], "metadata freeway")
    meta_pm_col = require_col(meta_raw, ["Abs PM", "abs_pm", "AbsPM", "Postmile", "PM"], "metadata absolute postmile")

    meta = meta_raw.rename(columns={
        meta_id_col: "station",
        meta_fwy_col: "Fwy",
        meta_pm_col: "Abs PM",
    }).copy()

    meta["station"] = pd.to_numeric(meta["station"], errors="coerce").astype("Int64")
    meta = meta.dropna(subset=["station"]).copy()
    meta["station"] = meta["station"].astype(int)

    def build_adjacency_fwy_dir(meta_df, stations, station_dir, k_neighbors=4):
        meta_sub = meta_df[meta_df["station"].isin(stations)].copy()
        meta_sub["fwy"] = meta_sub["Fwy"].astype(str)
        meta_sub["abs_pm"] = pd.to_numeric(meta_sub["Abs PM"], errors="coerce")
        meta_sub["direction"] = meta_sub["station"].map(station_dir)

        meta_sub = meta_sub.dropna(subset=["abs_pm", "direction"]).copy()
        meta_sub["direction"] = meta_sub["direction"].astype(str)

        station_to_idx = {s: i for i, s in enumerate(stations)}
        N_local = len(stations)
        A_dir = np.zeros((N_local, N_local), dtype=np.float32)

        all_dists = []
        for (_, _), grp in meta_sub.sort_values(["fwy", "direction", "abs_pm"]).groupby(["fwy", "direction"]):
            pm = grp["abs_pm"].values
            if len(pm) < 2:
                continue
            d = np.diff(np.sort(pm))
            d = d[d > 0]
            all_dists.extend(d.tolist())

        sigma = float(np.median(all_dists)) if len(all_dists) else 0.5
        sigma = max(sigma, 1e-3)

        def weight(dist):
            return float(np.exp(-(dist ** 2) / (sigma ** 2)))

        for (_, _), grp in meta_sub.sort_values(["fwy", "direction", "abs_pm"]).groupby(["fwy", "direction"]):
            grp = grp.sort_values("abs_pm")
            ids = grp["station"].astype(int).tolist()
            pms = grp["abs_pm"].astype(float).tolist()

            for i, sid in enumerate(ids):
                ii = station_to_idx[sid]
                for step in range(1, k_neighbors + 1):
                    if i - step >= 0:
                        sj = ids[i - step]
                        jj = station_to_idx[sj]
                        A_dir[ii, jj] = weight(abs(pms[i] - pms[i - step]))
                    if i + step < len(ids):
                        sj = ids[i + step]
                        jj = station_to_idx[sj]
                        A_dir[ii, jj] = weight(abs(pms[i] - pms[i + step]))

        np.fill_diagonal(A_dir, 1.0)
        A_dir = np.maximum(A_dir, A_dir.T)
        return A_dir

    A_dir = build_adjacency_fwy_dir(meta, stations, station_dir, k_neighbors=4)
    A_rw  = row_normalize_dense(A_dir)
    A_rwT = row_normalize_dense(A_dir.T)

    supports = [
        dense_to_sparse(A_rw, DEVICE),
        dense_to_sparse(A_rwT, DEVICE),
    ]

    print("Directed adjacency shape:", A_dir.shape)
    print("Directed adjacency density:", float((A_dir > 0).mean()))
    print("Support nnz:", [int(s._nnz()) for s in supports])

else:
    print("\nRaw files not found. Falling back to adjacency A from the strict dataset.")

    A_rw = row_normalize_dense(A.astype(np.float32))
    A_rwT = row_normalize_dense(A.astype(np.float32).T)

    supports = [
        dense_to_sparse(A_rw, DEVICE),
        dense_to_sparse(A_rwT, DEVICE),
    ]

    print("Fallback adjacency shape:", A.shape)
    print("Fallback adjacency density:", float((A > 0).mean()))
    print("Support nnz:", [int(s._nnz()) for s in supports])


class NConv(nn.Module):
    def forward(self, x, A_sp):
        B, C, N_local, T_local = x.shape
        x_r = x.permute(2, 0, 1, 3).reshape(N_local, -1).float()
        out = torch.sparse.mm(A_sp, x_r)
        out = out.reshape(N_local, B, C, T_local).permute(1, 2, 0, 3)
        return out.to(dtype=x.dtype)


class DiffusionGraphConv(nn.Module):
    def __init__(self, c_in, c_out, supports, order=1, dropout=0.0):
        super().__init__()
        self.nconv = NConv()
        self.supports = supports
        self.order = order
        self.dropout = dropout

        c_total = c_in * (1 + len(supports) * order)
        self.mlp = nn.Conv2d(c_total, c_out, kernel_size=(1, 1))

    def forward(self, x):
        out = [x]
        for A_sp in self.supports:
            x1 = self.nconv(x, A_sp)
            out.append(x1)
            for _ in range(2, self.order + 1):
                x1 = self.nconv(x1, A_sp)
                out.append(x1)

        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return h


class CausalConv2d(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=2, dilation=1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv2d(
            c_in,
            c_out,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
        )

    def forward(self, x):
        x = F.pad(x, (self.pad, 0, 0, 0))
        return self.conv(x)


class GraphWaveNetEncoder(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        supports,
        residual_channels=32,
        dilation_channels=32,
        skip_channels=64,
        end_channels=128,
        kernel_size=2,
        blocks=2,
        layers_per_block=4,
        gcn_order=1,
        dropout=0.1,
    ):
        super().__init__()
        self.dropout = dropout
        self.kernel_size = kernel_size
        self.blocks = blocks
        self.layers_per_block = layers_per_block

        receptive_field = 1
        for _ in range(blocks):
            for i in range(layers_per_block):
                receptive_field += (kernel_size - 1) * (2 ** i)
        self.receptive_field = receptive_field

        self.start_conv = nn.Conv2d(in_dim, residual_channels, kernel_size=(1, 1))

        self.filter_convs = nn.ModuleList()
        self.gate_convs   = nn.ModuleList()
        self.skip_convs   = nn.ModuleList()
        self.bn           = nn.ModuleList()
        self.gconvs       = nn.ModuleList()

        for _ in range(blocks):
            for i in range(layers_per_block):
                dilation = 2 ** i
                self.filter_convs.append(CausalConv2d(residual_channels, dilation_channels, kernel_size, dilation))
                self.gate_convs.append(CausalConv2d(residual_channels, dilation_channels, kernel_size, dilation))
                self.skip_convs.append(nn.Conv2d(dilation_channels, skip_channels, kernel_size=(1, 1)))
                self.gconvs.append(
                    DiffusionGraphConv(
                        dilation_channels,
                        residual_channels,
                        supports,
                        order=gcn_order,
                        dropout=dropout,
                    )
                )
                self.bn.append(nn.BatchNorm2d(residual_channels))

        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels, kernel_size=(1, 1))

    def forward(self, x):
        if x.size(-1) < self.receptive_field:
            pad_len = self.receptive_field - x.size(-1)
            x = F.pad(x, (pad_len, 0, 0, 0))

        x = self.start_conv(x)
        skip = None

        for i in range(len(self.filter_convs)):
            residual = x

            filt = torch.tanh(self.filter_convs[i](x))
            gate = torch.sigmoid(self.gate_convs[i](x))
            x = filt * gate
            x = F.dropout(x, p=self.dropout, training=self.training)

            s = self.skip_convs[i](x)
            skip = s if skip is None else (skip + s)

            x = self.gconvs[i](x)
            x = x + residual
            x = self.bn[i](x)

        x = F.relu(skip)
        x = F.relu(self.end_conv_1(x))
        return x


class GraphWaveNetRNN(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        out_len,
        supports,
        end_channels=128,
        use_gru=False,
        use_lstm=False,
        rnn_hidden=128,
        dropout=0.1,
        **encoder_kwargs,
    ):
        super().__init__()
        self.out_len = out_len
        self.use_gru = use_gru
        self.use_lstm = use_lstm

        self.encoder = GraphWaveNetEncoder(
            num_nodes=num_nodes,
            in_dim=in_dim,
            supports=supports,
            end_channels=end_channels,
            dropout=dropout,
            **encoder_kwargs,
        )

        if use_gru:
            self.gru = nn.GRU(input_size=end_channels, hidden_size=rnn_hidden, batch_first=True)
        else:
            self.gru = None

        if use_lstm:
            self.lstm = nn.LSTM(
                input_size=(rnn_hidden if use_gru else end_channels),
                hidden_size=rnn_hidden,
                batch_first=True,
            )
        else:
            self.lstm = None

        final_dim = rnn_hidden if (use_gru or use_lstm) else end_channels

        self.time_embed = nn.Linear(4, final_dim)
        self.horizon_out = nn.Linear(final_dim, 1)

    def forward(self, x, tf_future):
        h = self.encoder(x)
        B, C, N_local, T_local = h.shape

        h_seq = h.permute(0, 2, 3, 1).contiguous().view(B * N_local, T_local, C)

        if self.gru is not None:
            h_seq, _ = self.gru(h_seq)

        if self.lstm is not None:
            h_seq, _ = self.lstm(h_seq)

        last = h_seq[:, -1, :]
        z = last.view(B, N_local, -1)

        te = self.time_embed(tf_future)
        out = F.relu(z.unsqueeze(1) + te.unsqueeze(2))
        out = self.horizon_out(out).squeeze(-1)
        return out


COMMON_GWN_KWARGS = dict(
    num_nodes=N,
    in_dim=Fdim,
    out_len=OUT_LEN,
    supports=supports,
    end_channels=128,
    rnn_hidden=128,
    dropout=0.1,
    residual_channels=32,
    dilation_channels=32,
    skip_channels=64,
    kernel_size=2,
    blocks=2,
    layers_per_block=4,
    gcn_order=1,
)

xb, yb, tfb = next(iter(train_loader))
gwn_test_model = GraphWaveNetRNN(
    **COMMON_GWN_KWARGS,
    use_gru=False,
    use_lstm=False,
).to(DEVICE)

with torch.no_grad():
    out = gwn_test_model(xb.to(DEVICE), tfb.to(DEVICE))

print("GraphWaveNet sanity output:", out.shape)

del gwn_test_model
clear_memory()

Resolved TRAFFIC_CSV: /notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/cleaned_traffic_data.csv
Resolved META_XLSX: /notebooks/Spatio-Temporal-Prediction-and-Coordination-of-EV-Charging-Demand-for-Power-System-Resilience/pems_output.xlsx

Using refined raw-file graph support construction...
Directed adjacency shape: (1821, 1821)
Directed adjacency density: 0.0038374073179432942
Support nnz: [12720, 12720]
GraphWaveNet sanity output: torch.Size([8, 72, 1821])


In [21]:
gwn_model = GraphWaveNetRNN(
    **COMMON_GWN_KWARGS,
    use_gru=False,
    use_lstm=False,
).to(DEVICE)

run_dir_gwn, test_gwn = train_deep_model(
    model_name="GraphWaveNet",
    model=gwn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=40,
    patience=6,
    eval_every=2,
)

print("GraphWaveNet run dir:", run_dir_gwn)
del gwn_model
clear_memory()

Run dir: artifacts/runs/20260321_002039_GraphWaveNet_eval_h1_6_12_24_keep_out72


Train 1/40:   0%|          | 0/127 [00:00<?, ?it/s]

Train 2/40:   0%|          | 0/127 [00:00<?, ?it/s]

Eval:   0%|          | 0/37 [00:00<?, ?it/s]


Epoch 2: train_loss=0.113035 | val_avg_MAE=172.060

Validation
    1h | MAE=159.694 | RMSE=296.642
    6h | MAE=171.668 | RMSE=316.195
   12h | MAE=177.022 | RMSE=330.344
   24h | MAE=179.856 | RMSE=337.042


...


Epoch 32: train_loss=0.071076 | val_avg_MAE=124.229

Validation
    1h | MAE=107.130 | RMSE=198.487
    6h | MAE=119.489 | RMSE=224.249
   12h | MAE=128.483 | RMSE=257.220
   24h | MAE=141.814 | RMSE=283.407

Early stopping triggered. Best val_avg_MAE=121.665 at epoch 20.

Evaluating on test set...


Eval:   0%|          | 0/85 [00:00<?, ?it/s]


GraphWaveNet — TEST
    1h | MAE=118.054 | RMSE=233.117
    6h | MAE=130.607 | RMSE=256.106
   12h | MAE=132.210 | RMSE=262.220
   24h | MAE=132.825 | RMSE=266.317


Collect preds:   0%|          | 0/85 [00:00<?, ?it/s]

Saved outputs to: artifacts/runs/20260321_002039_GraphWaveNet_eval_h1_6_12_24_keep_out72
GraphWaveNet run dir: artifacts/runs/20260321_002039_GraphWaveNet_eval_h1_6_12_24_keep_out72
